### `1.实例`

In [1]:
def make_instance(cls):
    """Return a new object instance, which is a dispatch dictionary"""
    def get_value(name):
        if name in attributes:
            return attributes[name]
        else:
            value = cls['get'](name)
            return bind_method(value, instance)
    
    def set_value(name, value):
        attributes[name] = value
    
    attributes = {}
    instance = {'get': get_value, 'set': set_value}
    return instance

In [2]:
def bind_method(value, instance):
    """Return a bound method if value is callable, or value otherwise"""
    if callable(value):
        def method(*args):
            return value(instance, *args)
        return method
    else:
        return value

### `2.类`

In [3]:
def make_class(attributes, base_class=None):
    """Return a new class, which is a dispatch dictionary"""
    def get_value(name):
        if name in attributes:
            return attributes[name]
        elif base_class is not None:
            return base_class['get'](name)
    
    def set_value(name, value):
        attributes[name] = value
    
    def new(*args):
        return init_instance(cls, *args)
    
    cls = {'get': get_value, 'set': set_value, 'new': new}
    return cls

In [4]:
def init_instance(cls, *args):
    """Return a new object with type cls, initialized with args"""
    instance = make_instance(cls)
    init = cls['get']('__init__')
    if init:
        init(instance, *args)
    return instance

### `3.使用已经实现的对象`

In [6]:
def make_account_class():
    """Return the Account class, which has deposit and withdraw methods"""
    interest = 0.02

    def __init__(self, account_holder):
        self['set']('holder', account_holder)
        self['set']('balance', 0)
    
    def deposit(self, amount):
        new_balance = self['get']('balance') + amount
        self['set']('balance', new_balance)
        return self['get']('balance')
    
    def withdraw(self, amount):
        balance = self['get']('balance')
        if amount > balance:
            return 'Insufficient funds'
        new_balance = balance - amount
        self['set']('balance', new_balance)
        return self['get']('balance')
    return make_class(locals())

In [7]:
# Account类实例化
Account = make_account_class()

In [10]:
krik_account = Account['new']('krik')

In [11]:
krik_account['get']('deposit')(100)

100

In [18]:
def make_checking_account_class():
    """Return the CheckingAccount class, which imposes a $1 withdrawal fee."""
    interest = 0.01
    withdraw_fee = 1

    def withdraw(self, amount):
        fee = self['get']('withdraw_fee')
        return Account['get']('withdraw')(self, amount + fee)
    
    return make_class(locals(), Account)

In [19]:
CheckingAccount = make_checking_account_class()
jack_acct = CheckingAccount['new']('jack')

In [20]:
jack_acct['get']('interest')

0.01

In [21]:
jack_acct['get']('deposit')(100)

100

In [22]:
jack_acct['get']('withdraw')(50)

49